In [18]:
import numpy as np

In [19]:
### Test which of Quickhull or moment is the fastet
### Could run a large array with epsilon 

In [20]:
%load_ext cython

The cython extension is already loaded. To reload it, use:
  %reload_ext cython


In [21]:
%%cython --a
from scipy.spatial import voronoi_plot_2d
from scipy.spatial import Voronoi, ConvexHull
import numpy as np
def get_polygons_properties_2d(regions, vertices):
    """
    Extract the perimeter and area for each finite polygon in 2D
    Partially Cython typed.
    
    Parameters
    -----
    regions : numpy array
        Indices of vertices in each revised Voronoi regions.
    vertices : numpy array
        Coordinates for revised Voronoi vertices. Same as coordinates
        of input vertices, with 'points at infinity' appended to the
        end.
    
    Returns
    -----
    properties_index : numpy array
        List of index for each region in regions treated (probably not useful since perimeter and area are
        already acquired in the order of region in regions)
    properties_perimeter : numpy array
        Perimeter of each region in regions treated
    properties_area : numpy array
        Area of each region in regions treated
        Warning : In 2D : ConvexHull.area is the perimeter, while ConvexHull.volume is the area
    """
    
    ## Typing variables
    len_regions = len(regions)
    
    properties_index_np = np.zeros(len_regions)
    properties_perimeter_np = np.zeros(len_regions)
    properties_area_np = np.zeros(len_regions)
    
    properties_index = properties_index_np
    properties_perimeter = properties_perimeter_np
    properties_area = properties_area_np

    for index in range(len_regions):
        polygon = vertices[regions[index]]
        properties_index[index] = index
        convHull = ConvexHull(polygon)
        convHull_area = convHull.area
        convHull_volume = convHull.volume
        properties_perimeter[index] = convHull_area
        properties_area[index] = convHull_volume
        
    return properties_index,properties_perimeter,properties_area



###############

def compute_moment_tensors_full(regions, vertices, points, areas):
    """
    Compute a moment tensor for every cell (simple polygons, 0th, 1st and 2nd moments)
    Based on Mulchrone and Choudhury, J. of Structural Geology 26 (2004)
    
    Parameters:
    -----
    regions : numpy array 
        Array of arrays pointing to the "vertices" array, determining a cell
    vertices : numpy array
        Coordinates of cell vertices
    points : numpy array
        Coordinates of Voronoi centers
    areas : numpy array
        Areas of Voronoi cells

    Returns:
    -----
    moment_tensors : numpy array 
        Array of 3X3 tensors for every cell
    """
    moment_tensors = np.zeros([len(regions), 3, 3])
    #centroid_x = points[:,0] 
    #centroid_y = points[:,1]

    # DEBUG
    # print("centroid_x : \n{}".format(centroid_x))
    # print("centroid_y : \n{}".format(centroid_y))
    tmp = 0
    # Mnemonic for easier manipulation
    for index, region in enumerate(regions):
        # Load the numpy arrays
        x_i = vertices[region, 0]
        y_i = vertices[region, 1]

        # use np.roll to have a "+1" array that wraps at the end
        x_i_p1 = np.roll(x_i, -1) 
        y_i_p1 = np.roll(y_i, -1) 

        # cu : cube ; sq : square
        x_i_cu = x_i * x_i * x_i
        x_i_sq = x_i * x_i 
        x_i_p1_cu = x_i_p1 * x_i_p1 * x_i_p1
        x_i_p1_sq = x_i_p1 * x_i_p1 
        y_i_cu = y_i * y_i * y_i
        y_i_sq = y_i * y_i 
        y_i_p1_cu = y_i_p1 * y_i_p1 * y_i_p1
        y_i_p1_sq = y_i_p1 * y_i_p1
    
        # Need to find a way to sum all the points' coordinates within a region 
        # m_00 = area (assumed already computed with Quick Hull)
        moment_tensors[index, 0, 0] = np.sum((x_i + x_i_p1) * (y_i - y_i_p1), axis=0)
        # m_01 
#        moment_tensors[index, 0, 1] = centroid_y[index] * areas[index]
        moment_tensors[index, 0, 1] = np.sum((x_i - x_i_p1) * (y_i_sq + y_i * y_i_p1 + y_i_p1_sq) , axis=0)
        # m_10
#        moment_tensors[index, 1, 0] = centroid_x[index] * areas[index]   
        moment_tensors[index, 1, 0] = np.sum( (x_i_sq + x_i * x_i_p1 + x_i_p1_sq) * (y_i - y_i_p1), axis=0)
#        print("m10 = {}".format(moment_tensors[index, 1, 0]))
        tmp += moment_tensors[index, 1, 0]
#        print("tmp = {}".format(tmp))
        
        # m_02
        moment_tensors[index, 0, 2] = np.sum( 
            (x_i - x_i_p1) * (y_i_cu + y_i_sq * y_i_p1 + y_i * y_i_p1_sq + y_i_p1_cu), axis=0)
        
        # m_20
        moment_tensors[index, 2, 0] = np.sum( 
            (x_i_cu + x_i_sq * x_i_p1 + x_i * x_i_p1_sq + x_i_p1_cu ) * (y_i - y_i_p1), axis=0)
        
        # m_11
        moment_tensors[index, 1, 1] = np.sum( 
        (x_i - x_i_p1) * (x_i * (3*y_i_sq + 2*y_i*y_i_p1 + y_i_p1_sq) 
                            + x_i_p1 * (y_i_sq + 2*y_i*y_i_p1 + 3*y_i_p1_sq)), axis=0)
 
    
        
    # m_12
    # Not computed
    # m_21
    # Not computed
    # m_22
    # Not computed
    
    # Do the divisions once at the end
    moment_tensors[:, 0, 0] = moment_tensors[:, 0, 0] * (-1.0/2.0)                     
    moment_tensors[:, 0, 1] = moment_tensors[:, 0, 1] * (1.0/6.0)
    moment_tensors[:, 1, 0] = moment_tensors[:, 1, 0] * (-1.0/6.0)
    moment_tensors[:, 1, 1] = moment_tensors[:, 1, 1] * (1.0/24.0)                  
    moment_tensors[:, 0, 2] = moment_tensors[:, 0, 2] * (1.0/12.0)   
    moment_tensors[:, 2, 0] = moment_tensors[:, 2, 0] * (-1.0/12.0)   
#    moment_tensors[:, 2, 0] = moment_tensors[:, 2, 0] * (1.0/12.0)      
    # Ensure that the Ixx and Iyy are positive
#    moment_tensors[:, 0, 0] = np.abs(moment_tensors[:, 0, 0])
#    moment_tensors[:, 1, 1] = np.abs(moment_tensors[:, 1, 1])   
    return moment_tensors 




In [22]:
def generate_hexagonal_seed(point_number, boxsize, lattice_noise=0):
    """
    Create seed points (Voronoi center) on a controlled or "noisy" (with a bit of noise) hexagonal lattice, and applies periodic boundary conditions.
    TO DO: initialize polarity, angles, momentum, a flag indicating at which stage we are currently in.
    
    Parameters
    -----
    point_number : int
        Number of seed point to create
    boxsize : float
        Length of the axis for observation box (assumed square for now)
    lattice_noise : float
        Intensity of the noise applied to the lattice seed points

    Returns
    -----
    generated_points : ndarray of ndarray
        Array of seed points on a lattice (of coordinates x,y)
    generated_angles : ndarray
        Array of randomly drawn angles in radians
    generated_polarities : ndarray of double
        Array of polarity vectors (cos theta_i, sin theta_i), initialized with random initial values 
    (in the future maybe): flag
    """
    # FIXME Step should be computed and not hard coded
    cpt = 0
    generated_points = np.zeros([int(boxsize)**2,2])

    for y in np.arange(-boxsize/2.0,boxsize/2.0,1):
        if(y % 2 == 0):
            for x in np.arange(-boxsize/2.0,boxsize/2.0,1):
                generated_points[cpt][0] = x
                generated_points[cpt][1] = y               
                cpt = cpt + 1
        else:
            for x in np.arange( (-boxsize/2.0)+0.5,boxsize/2.0,1):
                generated_points[cpt][0] = x
                generated_points[cpt][1] = y               
                cpt = cpt + 1

#    #### XXX : I do not see the point of doing this in the initialization part ####
#    for coord in generated_points:
#        coord[0] = generate_periodic_boundaries(coord[0], boxsize) # x coordinate
#        coord[1] = generate_periodic_boundaries(coord[1], boxsize) # y coordinate
#        # print(coord[0], coord[1])
#    
#
#
#    # ADD ANGLE INITIALISATION, POLARITY, NOISE, ETC.
#    # Angle initialisation for every seed point (initialisation is done in radians)
#    generated_angles = np.random.random(len(generated_points))* 2.0 * np.pi
#    # Generate the associated polarity vectors
#    generated_polarities = generate_every_polarity_vector(generated_angles)
#
    # XXX Temporary return for prototyping
    return generated_points


def hexagonal_lattice_initialization(point_number, boxsize, lattice_noise=0):
    """
    Create seed points (Voronoi center) on an triangular pattern (to give hexagonal cells) and applies periodic boundary conditions.
    TO DO: initialize polarity, angles, momentum, a flag indicating at which stage we are currently in.
    
    Parameters
    -----
    point_number : int
        Number of seed point to create
    boxsize : float
        Length of the axis for observation box (assumed square for now)
    lattice_noise : float
        Intensity of the noise applied to the lattice seed points

    Returns
    -----
    generated_points : ndarray of ndarray
        Array of seed points on a lattice (of coordinates x,y)
    generated_angles : ndarray
        Array of randomly drawn angles in radians
    generated_polarities : ndarray of double
        Array of polarity vectors (cos theta_i, sin theta_i), initialized with random initial values 
    (in the future maybe): flag
    """
    # FIXME Step should be computed and not hard coded
    generated_points = generate_hexagonal_seed(point_number, boxsize, lattice_noise=0)


    # Extremely important : places everyone in the observation box
    # TODO : Maybe find a better name for the function
    for coord in generated_points:
        coord[0] = generate_periodic_boundaries(coord[0], boxsize) # x coordinate
        coord[1] = generate_periodic_boundaries(coord[1], boxsize) # y coordinate
        # print(coord[0], coord[1])
    

    # ADD ANGLE INITIALISATION, POLARITY, NOISE, ETC.
    # Angle initialisation for every seed point (initialisation is done in radians)
    generated_angles = np.random.random(len(generated_points))* 2.0 * np.pi
    # Generate the associated polarity vectors
#    generated_polarities = generate_every_polarity_vector(generated_angles)
    generated_polarities = np.zeros(len(generated_points))
    return generated_points, generated_angles, generated_polarities



In [23]:
def voronoi_finite_polygons_2d(vor, radius=None):
    """
    Reconstruct infinite voronoi regions in a 2D diagram to finite regions.

    Parameters
    ----------
    vor : Voronoi
        Input diagram
    radius : float, optional
        Distance to 'points at infinity'.

    Returns
    -------
    regions : list of lists
        Indices of vertices in each revised Voronoi regions.
        Vertices are sorted counter-clockwise.
    vertices : list of lists
        Coordinates for revised Voronoi vertices. Same as coordinates
        of input vertices, with 'points at infinity' appended to the
        end.

    """

    if vor.points.shape[1] != 2:
        raise ValueError("Requires 2D input")

    new_regions = []
    new_vertices = vor.vertices.tolist()

    center = vor.points.mean(axis=0)
    if radius is None:
        radius = vor.points.ptp().max()*2

    # Construct a map containing all ridges for a given point
    all_ridges = {}
    for (p1, p2), (v1, v2) in zip(vor.ridge_points, vor.ridge_vertices):
        all_ridges.setdefault(p1, []).append((p2, v1, v2))
        all_ridges.setdefault(p2, []).append((p1, v1, v2))

    # Reconstruct infinite regions
    for p1, region in enumerate(vor.point_region):
        vertices = vor.regions[region]

        if all(v >= 0 for v in vertices):
            # finite region
            new_regions.append(vertices)
            continue

        # reconstruct a non-finite region
        ridges = all_ridges[p1]
        new_region = [v for v in vertices if v >= 0]

        for p2, v1, v2 in ridges:
            if v2 < 0:
                v1, v2 = v2, v1
            if v1 >= 0:
                # finite ridge: already in the region
                continue

            # Compute the missing endpoint of an infinite ridge

            t = vor.points[p2] - vor.points[p1] # tangent
            t /= np.linalg.norm(t)
            n = np.array([-t[1], t[0]])  # normal

            midpoint = vor.points[[p1, p2]].mean(axis=0)
            direction = np.sign(np.dot(midpoint - center, n)) * n
            far_point = vor.vertices[v2] + direction * radius

            new_region.append(len(new_vertices))
            new_vertices.append(far_point.tolist())


        # sort region counterclockwise
        vs = np.asarray([new_vertices[v] for v in new_region])
        c = vs.mean(axis=0)
        angles = np.arctan2(vs[:,1] - c[1], vs[:,0] - c[0])
        new_region = np.array(new_region)[np.argsort(angles)]

        # finish
        new_regions.append(new_region.tolist())

    return new_regions, np.asarray(new_vertices)

def generate_periodic_boundaries(val, box):
    """
    Generates periodic boundaries by translating values to fit inside the observation box.
    
    Inspired by Hiraiwa Tetsuya. Modified because of performance issues. 
    Parameters:
        val: double, coordinate
        box: size of one side of the observation box
    -----
    Returns:
        translated_value: double, coordinate after translation
    
    Note: in CCM.C (program by T.H), this function is called at initialization and [after Runge-Kutta but before polarity time evolution]
    """

    ## Just like in a videogame, when you cross a side of the screen, you appear back on the other side

    # while(val > box/2.0):
    #     # val = val - box
    #     # The above option is correct but time consuming
    #     val = val % box
    # while( val <= -box/2.0):
    #     # val = val + box
    #     # The above option is correct but time consuming
    #     val = val % box

    while(val < -box/2.0 or val > box/2.0):
        if(val > box/2.0 and val < box):
            val = (box % val) * (-1.0)
        elif(val < -box/2.0 and val > -box):
            val = (-box % val) * (-1.0)
        elif(val >= box):
            val = val % box
        elif(val <= box):
            val = val % -box
    return val

def generate_periodic_voronoi_cells_2( points, boxsize):
    """
    Copy the initialized points around the observation box of size "boxsize", over the 8 nearest regions
    C-like version (250+ times faster than the pure python)

    Parameters:
    points : numpy array of arrays
        (x,y) coordinates of the Voronoi centers
    boxsize : float
        Size of a side of the observation box (assumed square-shaped)
    -----
    Returns:
    extended_points : numpy array of arrays
        (x,y) coordinates of the Voronoi centers and their copy
    """

    #Define all the variables
    len_points = len(points)
    print(type(len_points))
    DTYPE=np.float64
    # Empty point array to store the original points and three copies
    extended_points_np = np.zeros((len(points)*9,2), dtype=DTYPE)
    extended_points = extended_points_np

    index = 0
    # Copy the original values
    for index in range(len_points):
        extended_points[index][0] = points[index][0]
        extended_points[index][1] = points[index][1]

        # Translate along the y-axis
    for index in range(len_points):
        extended_points[index+len_points][0] = points[index][0]
        extended_points[index+len_points][1] = points[index][1] + boxsize
        extended_points[index+len_points*2][0] = points[index][0]
        extended_points[index+len_points*2][1] = points[index][1] - boxsize

    # Translate along the x-axis
    for index in range(len_points):
        extended_points[index+len_points*3][0] = points[index][0] + boxsize
        extended_points[index+len_points*3][1] = points[index][1]
        extended_points[index+len_points*4][0] = points[index][0] - boxsize
        extended_points[index+len_points*4][1] = points[index][1]

    # Translate along the diagonal
    for index in range(len_points):
        extended_points[index+len_points*5][0] = points[index][0] + boxsize
        extended_points[index+len_points*5][1] = points[index][1] + boxsize

        extended_points[index+len_points*6][0] = points[index][0] - boxsize
        extended_points[index+len_points*6][1] = points[index][1] - boxsize

        extended_points[index+len_points*7][0] = points[index][0] + boxsize
        extended_points[index+len_points*7][1] = points[index][1] - boxsize

        extended_points[index+len_points*8][0] = points[index][0] - boxsize
        extended_points[index+len_points*8][1] = points[index][1] + boxsize

    return extended_points




In [24]:
def initialization(point_number, boxsize):
    """
    Create an array of random seed points (Voronoi center), and applies periodic boundary conditions.
    TO DO: initialize polarity, angles, momentum, a flag indicating at which stage we are currently in.
    
    Parameters
    -----
    point_number : int
        Number of seed point to create
    boxsize : float 
        Length of the axis for observation box (assumed square for now)

    Returns
    -----
    generated_points : ndarray of ndarray
        Array of random seed points (of coordinates x,y)
    generated_angles : ndarray
        Array of randomly drawn angles in radians
    generated_polarities : ndarray of double
        Array of polarity vectors (cos theta_i, sin theta_i), initialized with random initial values 
    (in the future maybe): flag
    """
    # XXX : DEBUG
    generated_points = np.random.rand(point_number, 2) * boxsize - (boxsize * 0.5 * 0.5)
    # XXX : With a Poisson distribution
    # generated_points = np.random.poisson(lam=1.0,size=(point_number, 2)) * boxsize - (boxsize * 0.5 * 0.5)
    # generated_points = (2* np.random.rand(point_number, 2) - 1) * boxsize * 0.5
    # DEBUG 
    # generated_points = generated_points * 10
    # print(generated_points)
    # print('-'*10)

    # Extremely important : places everyone in the observation box
    # TODO : Maybe find a better name for the function
    for coord in generated_points:
        coord[0] = generate_periodic_boundaries(coord[0], boxsize) # x coordinate
        coord[1] = generate_periodic_boundaries(coord[1], boxsize) # y coordinate
        # print(coord[0], coord[1])
    


    # ADD ANGLE INITIALISATION, POLARITY, NOISE, ETC.
    # Angle initialisation for every seed point (initialisation is done in radians)
    generated_angles = np.random.random(len(generated_points))* 2.0 * np.pi
    # Generate the associated polarity vectors
#    generated_polarities = generate_every_polarity_vector(generated_angles)
    generated_polarities = np.zeros(len(generated_points))
    return generated_points, generated_angles, generated_polarities


In [25]:
boxsize = 10
#np.random.seed(29308)
np.random.seed(4876)
# points = hexagonal_lattice_initialization(100,boxsize)[0]
points = initialization(100,boxsize)[0]
extended_points = generate_periodic_voronoi_cells_2(points,boxsize)
vor = Voronoi(extended_points)
regions, vertices = voronoi_finite_polygons_2d(vor)


<class 'int'>


In [26]:
%timeit get_polygons_properties_2d(regions, vertices)

114 ms ± 9.59 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [27]:
%timeit compute_moment_tensors_full(regions, vertices, None, None)

112 ms ± 1.52 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [28]:
perimeters, areas = get_polygons_properties_2d(regions, vertices)[1:3]

In [29]:
moment_tensors = compute_moment_tensors_full(regions, vertices, None, None)

In [30]:
diff = areas[0:len(points)] - np.abs(moment_tensors[0:len(points),0,0])

In [31]:
np.mean(diff)

-1.0269562977782698e-17

In [32]:
np.abs(moment_tensors[0:100,0,0])

array([0.92851241, 0.65805286, 0.79571422, 2.35376566, 0.45612114,
       0.91935723, 0.17059391, 0.7673502 , 3.03235769, 0.56279112,
       0.6187727 , 0.99498446, 0.546457  , 0.78959168, 1.31142139,
       1.57139169, 1.31349319, 1.36689948, 1.64726532, 0.7629687 ,
       0.78162922, 0.29472585, 0.64681032, 1.67167947, 0.54563079,
       0.88051438, 0.79332705, 1.76982262, 0.39902265, 0.71903266,
       0.91658383, 1.23412238, 0.44671598, 0.41632234, 0.83150324,
       1.87042025, 1.4387983 , 0.98252352, 1.01152801, 1.6546722 ,
       1.10051483, 0.77078514, 0.82791852, 1.76628168, 1.17049097,
       0.28232212, 0.82932536, 0.4138859 , 0.89348856, 0.65207413,
       1.03382974, 1.85048818, 1.58175499, 0.83557749, 0.42097362,
       1.20170449, 0.12656081, 1.16340318, 0.67803519, 1.14026276,
       1.21808709, 1.19699732, 0.86742014, 1.10565196, 1.92469792,
       0.35098296, 1.96627068, 0.93894954, 1.36484284, 1.88807806,
       1.96984706, 1.00499751, 0.54530057, 0.31600762, 0.57216

In [93]:
def compute_perimeters_areas(regions, vertices):
    """
    #TODO
    """
    
    areas = np.zeros(len(regions))
    perimeters = np.zeros(len(regions))
    for index, region in enumerate(regions):
        # Load the numpy arrays
        x_i = vertices[region, 0]
        y_i = vertices[region, 1]
        coordinates_vectors = np.zeros([len(x_i),2])
        coordinates_vectors[:,0] = x_i
        coordinates_vectors[:,1] = y_i
        # use np.roll to have a "+1" array of next indices that wraps at the end
        x_i_p1 = np.roll(x_i, -1) 
        y_i_p1 = np.roll(y_i, -1) 

        ### Compute the areas (zeroth moment tensor) ###
        areas[index] = np.sum((x_i + x_i_p1) * (y_i - y_i_p1), axis=0)
        
        ### Compute the perimeters ###
        coordinates_vectors[:,0] = coordinates_vectors[:,0] - x_i_p1
        coordinates_vectors[:,1] = coordinates_vectors[:,1] - y_i_p1
#        print(coordinates_vectors)
#        print(np.linalg.norm(coordinates_vectors, axis=1))
        perimeters[index] = np.sum(np.linalg.norm(coordinates_vectors, axis=1))
    
    # Do all the division in one go at the end
    areas = areas * -0.5                     
    
    return perimeters, areas

In [94]:
perimeters, areas = compute_perimeters_areas(regions, vertices)
ind, prop_perimeters, prop_areas = get_polygons_properties_2d(regions, vertices)
# Check if the result returned is coherent with the zeroth moment tensor
#print(areas==moment_tensors[:,0,0])
#print(perimeters - get_polygons_properties_2d(regions,vertices)[1])

In [95]:
## Informal comparison between the two methods
print("Without ConvHull : perimeters[0:10] = \n{}".format(perimeters[0:10]))
print("With ConvHull : prop_perimeters[0:10] = \n{}".format(prop_perimeters[0:10]))
diff = perimeters - prop_perimeters

print("Difference perimeters - prop_perimeters")
print('-'*10)
print("Max diff : {}".format(np.max(diff)))
print("Min diff : {}".format(np.min(diff)))

Without ConvHull : perimeters[0:10] = 
[4.07147701 3.20884039 3.62070017 6.13246063 3.03752503 3.941223
 1.82535095 4.03586021 6.8902498  2.9556283 ]
With ConvHull : prop_perimeters[0:10] = 
[4.07147701 3.20884039 3.62070017 6.13246063 3.03752503 3.941223
 1.82535095 4.03586021 6.8902498  2.9556283 ]
Difference perimeters - prop_perimeters
----------
Max diff : 9.094947017729282e-13
Min diff : -2.2737367544323206e-13


In [96]:
print(1.10528784+0.33189175+0.24992891+0.88327743+1.33361217+0.1674789)
tab = np.array([1.10528784,0.33189175,0.24992891,0.88327743,1.33361217,0.1674789,])
print(np.sum(tab))
print(0.30138435+0.63664576+1.02212294+0.29898047+0.26288262+0.68682425)
print(prop_perimeters)

4.071477
4.071477
3.2088403899999998
[4.07147701e+00 3.20884039e+00 3.62070017e+00 6.13246063e+00
 3.03752503e+00 3.94122300e+00 1.82535095e+00 4.03586021e+00
 6.89024980e+00 2.95562830e+00 3.30432828e+00 4.07132728e+00
 3.15372957e+00 4.64529702e+00 4.55090728e+00 5.39016222e+00
 4.63869007e+00 4.99606869e+00 5.20429023e+00 3.68266511e+00
 4.13537622e+00 3.06268578e+00 3.17355718e+00 5.02175865e+00
 3.37396269e+00 3.74167278e+00 3.63087524e+00 5.17944683e+00
 3.22320912e+00 3.63874121e+00 4.17608528e+00 4.53156403e+00
 2.80541152e+00 3.12834363e+00 3.66194370e+00 5.31924545e+00
 4.78418327e+00 4.11472034e+00 4.25503632e+00 5.65327487e+00
 4.48259435e+00 3.74325101e+00 3.60649099e+00 5.10155925e+00
 4.22990061e+00 2.49379267e+00 3.61459367e+00 2.96821256e+00
 3.98393810e+00 3.52083990e+00 4.23754712e+00 5.29644371e+00
 5.52424421e+00 3.60154612e+00 2.71415678e+00 5.24021160e+00
 1.44882576e+00 4.92450541e+00 3.66240917e+00 4.47139717e+00
 4.20174309e+00 4.37881133e+00 4.10060178e+00 4.

In [97]:
%timeit compute_perimeters_areas(regions,vertices)

79.5 ms ± 4.55 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [112]:
%%cython --a
import numpy as np
cimport numpy as np
DTYPE = np.double
ctypedef np.double_t DTYPE_t
def compute_perimeters_areas_cython(regions, vertices):
    """
    #TODO
    """
    
    
    cdef int len_regions = len(regions)
    cdef int len_vector
    cdef np.ndarray areas = np.zeros(len_regions, dtype=DTYPE)
    cdef np.ndarray perimeters = np.zeros(len_regions, dtype=DTYPE)
    
#    cdef double[:] areas = areas_np
#    cdef double[:] perimeters = perimeters_np     
    cdef np.ndarray x_i
    cdef np.ndarray y_i
    cdef np.ndarray coordinates_vectors
    cdef np.ndarray x_i_p1
    cdef np.ndarray y_i_p1
 
    for index, region in enumerate(regions):
        # Load the numpy arrays
        len_vector = len(vertices[region])
        x_i = np.zeros(len_regions, dtype=DTYPE)
        y_i = np.zeros(len_regions, dtype=DTYPE)
        
        x_i = vertices[region, 0]
        y_i = vertices[region, 1]
        
        coordinates_vectors = np.zeros([len(x_i),2])
        coordinates_vectors[:,0] = x_i
        coordinates_vectors[:,1] = y_i
        # use np.roll to have a "+1" array of next indices that wraps at the end
        x_i_p1 = np.roll(x_i, -1) 
        y_i_p1 = np.roll(y_i, -1) 

        ### Compute the areas (zeroth moment tensor) ###
        areas[index] = np.sum((x_i + x_i_p1) * (y_i - y_i_p1), axis=0)
        
        ### Compute the perimeters ###
        coordinates_vectors[:,0] = coordinates_vectors[:,0] - x_i_p1
        coordinates_vectors[:,1] = coordinates_vectors[:,1] - y_i_p1
#        print(coordinates_vectors)
#        print(np.linalg.norm(coordinates_vectors, axis=1))
        perimeters[index] = np.sum(np.linalg.norm(coordinates_vectors, axis=1))
    # Do all the division in one go at the end
    areas = areas * (-0.5)                     
    return perimeters, areas

In [113]:
%timeit compute_perimeters_areas_cython(regions, vertices)

104 ms ± 5.17 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
